# PySpark dataset setup
Run this notebook first to validate your local environment.


> **Realistic Dataset v2:** Shared across Python, Pandas, and PySpark. Existing columns retained. Synthetic data includes intentional dirty records and malformed JSON for interview exercises.


In [1]:
import sys
print(sys.version)

In [2]:
import os, sys
# Use the active kernel's Python for local Spark workers; restart the kernel
# if Spark was previously started with another interpreter.
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, Window, types as T
from IPython.display import display

# Run from the extracted project directory in VS Code/Jupyter.
BASE = Path.cwd()
# Check bundled inputs before loading; never regenerate or overwrite fixtures.
_fixture_dir = BASE / 'data'
_required_fixtures = [
    f'{name}.{extension}'
    for name in ['customers', 'products', 'orders', 'order_items', 'events',
                 'transactions', 'pipeline_logs', 'cdc_changes']
    for extension in ['csv', 'json', 'jsonl']
] + ['config.json', 'customers_dirty.json', 'events_duplicates.jsonl',
     'malformed_events.jsonl']
_missing_fixtures = [name for name in _required_fixtures
                     if not (_fixture_dir / name).is_file()]
if _missing_fixtures:
    raise FileNotFoundError(
        f"Missing required fixture files in {_fixture_dir.resolve()}: "
        + ', '.join(_missing_fixtures)
        + ". Run from the pyspark/ directory with its bundled data/ folder, "
          "and obtain any missing files from the approved repository dataset. "
          "Setup does not generate or overwrite fixtures."
    )
spark = (SparkSession.builder.master('local[2]').config('spark.pyspark.python', sys.executable)
       .config('spark.pyspark.driver.python', sys.executable).appName('PySpark DE Interview Practice')
         .config('spark.sql.session.timeZone','UTC')
         .config('spark.sql.shuffle.partitions','8')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

csv_tables = ['customers','products','orders','order_items','transactions','pipeline_logs','cdc_changes']
for name in csv_tables:
    globals()[name] = (spark.read.option('header',True).option('inferSchema',True)
                       .csv(str(BASE / 'data' / (name+'.csv'))))
# Explicit nested JSON input (avoid treating nested properties as strings).
events = spark.read.json(str(BASE / 'data' / 'events.jsonl'))
# Corrupt records are preserved in a string column; explicit schema is important.
malformed_events = (spark.read.schema(T.StructType([
    T.StructField('event_id',T.LongType()),
    T.StructField('event_type',T.StringType()),
    T.StructField('_corrupt_record',T.StringType())
])).option('mode','PERMISSIVE').option('columnNameOfCorruptRecord','_corrupt_record')
.json(str(BASE / 'data' / 'malformed_events.jsonl')))
print('Spark version:',spark.version)
for name in csv_tables+['events']:
    print(f'{name}: {globals()[name].count()} rows')


In [3]:
orders.printSchema()
orders.show(5, truncate=False)
events.printSchema()


In [4]:
# Existing fixtures are intentionally left unchanged, including dirty records.
print('Dataset setup complete; existing fixture files were not modified.')
